## 라이브러리 임포트

분석에 필요한 라이브러리를 불러옵니다.

In [6]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

## 데이터 로드 및 파생 컬럼 생성

`steam_indie_list.csv` 원본 데이터를 불러오고, 분석에 필요한 파생 컬럼을 생성합니다.

- `total_reviews`: 긍정 + 부정 리뷰 합산
- `owners_lower`: `owners` 범위 문자열에서 하한값 추출
- `is_f2p`: 무료 플레이 여부
- `is_early_access`: 얼리 액세스 여부

In [7]:
df = pd.read_csv('../../../data/raw/steamspy_indie_games.csv')

df['total_reviews'] = df['positive'] + df['negative']
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')

def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except Exception:
        return []

df['genres']       = df['genres'].apply(parse_genres)

print(f'원본: {len(df):,}개')
print(f'Early Access: {df["genres"].apply(lambda gl: "Early Access" in gl).sum():,}개 ({df["genres"].apply(lambda gl: "Early Access" in gl).mean():.1%})')
print(f'Free To Play: {df["genres"].apply(lambda gl: "Free To Play" in gl).sum():,}개 ({df["genres"].apply(lambda gl: "Free To Play" in gl).mean():.1%})')

원본: 61,266개
Early Access: 6,340개 (10.3%)
Free To Play: 3,446개 (5.6%)


## 분석 대상 필터링

다음 조건을 모두 만족하는 게임을 메인 분석 모집단으로 선별합니다.

- 출시연도: 2023 ~ 2025년
- 리뷰 수: 10개 이상
- Early Access 제외
- Free to Play 제외

Early Access와 F2P 게임은 별도 데이터프레임(`df_ea`, `df_f2p`)으로 보존합니다.

In [8]:
MIN_REVIEWS = 10

is_ea  = df['genres'].apply(lambda gl: 'Early Access' in gl)
is_f2p = df['genres'].apply(lambda gl: 'Free To Play' in gl)

df_ea  = df[is_ea].copy()
df_f2p = df[~is_ea & is_f2p].copy()
df_f   = df[
    (df['total_reviews'] >= MIN_REVIEWS) &
    (df['release_date'].dt.year >= 2023) &
    (df['release_date'].dt.year <= 2025) &
    (~is_ea) &
    (~is_f2p)
].copy()

print(f'전체              : {len(df):,}개')
print(f'Early Access 제외 : {len(df_ea):,}개 → 별도 분석')
print(f'F2P 제외          : {len(df_f2p):,}개 → 별도 분석')
print(f'메인 모집단       : {len(df_f):,}개  (2023~2025년, 리뷰 {MIN_REVIEWS}개 이상, EA·F2P 제외)')
print(f'\n출시연도 분포 (메인):')
print(df_f['release_date'].dt.year.value_counts().sort_index().to_string())

전체              : 61,266개
Early Access 제외 : 6,340개 → 별도 분석
F2P 제외          : 3,064개 → 별도 분석
메인 모집단       : 9,692개  (2023~2025년, 리뷰 10개 이상, EA·F2P 제외)

출시연도 분포 (메인):
release_date
2023    3498
2024    4180
2025    2014


## 컬럼 정제

분석에 적합한 형태로 컬럼명을 정리하고 불필요한 컬럼을 제거합니다.

- `name`: `name_store` 우선, 없으면 `spy_name` 사용
- `price`: `price_spy` 컬럼명 변경
- `release_date`: `yyyy-MM-dd` 포맷으로 통일
- 불필요 컬럼(`spy_name`, `name_store`) 제거

In [9]:
# ── name_store 값으로 name_spy 대체 후 name 으로 컬럼명 변경 ─────────────────
df_f['name'] = df_f['name_store'].fillna(df_f['spy_name'])

# ── price_spy → price 컬럼명 변경 ────────────────────────────────────────────
df_f = df_f.rename(columns={'price_spy': 'price'})

# ── release_date → yyyy-MM-dd 포맷 ───────────────────────────────────────────
df_f['release_date'] = df_f['release_date'].dt.strftime('%Y-%m-%d')

# ── 불필요 컬럼 제거 ──────────────────────────────────────────────────────────
df_f = df_f.drop(columns=['spy_name', 'name_store', 'type'])

print('전처리 완료')
print(df_f[['name', 'release_date', 'price']].head())

전처리 완료
                              name release_date  price
10                      Last Epoch   2024-02-21   3499
23                   7 Days to Die   2024-07-25   4499
24                       CyberCorp   2025-04-22   1499
25              Sons Of The Forest   2024-02-22   2999
28  Warhammer 40,000: Rogue Trader   2023-12-07   4999


## steam_indie_review_summary 병합

- `total_reviews`, `positive`, `negative` → `steam_indie_review_summary` 값으로 대체
- `review_score`, `review_score_desc` 컬럼 추가

In [10]:
reviews = pd.read_csv('../../../data/raw/steam_indie_review_summary.csv')

common_cols_reviews = sorted(set(df_f.columns) & set(reviews.columns))
print('games vs review_summary 공통 컬럼:', common_cols_reviews)

replace_cols = reviews[['appid', 'total_reviews', 'total_positive', 'total_negative']]
extra_cols   = reviews.drop(columns=['total_reviews', 'total_positive', 'total_negative'])

df_f = df_f.merge(replace_cols, on='appid', how='left', suffixes=('_old', ''))
df_f['total_reviews'] = df_f['total_reviews'].combine_first(df_f['total_reviews_old'])
df_f['positive']      = df_f['total_positive'].combine_first(df_f['positive'])
df_f['negative']      = df_f['total_negative'].combine_first(df_f['negative'])
df_f = df_f.drop(columns=['total_reviews_old', 'total_positive', 'total_negative'])

df_f = df_f.merge(extra_cols, on='appid', how='left')

print('병합 결과 shape:', df_f.shape)
print('컬럼:', df_f.columns.tolist())

games vs review_summary 공통 컬럼: ['appid', 'total_reviews']
병합 결과 shape: (9692, 13)
컬럼: ['appid', 'owners', 'positive', 'negative', 'price', 'ccu', 'genres', 'release_date', 'developers', 'name', 'total_reviews', 'review_score', 'review_score_desc']


## steam_app_details 병합

- 공통 컬럼 (`name`, `developers`, `genres`, `release_date`) → `steam_app_details` 값으로 대체
- 나머지 `steam_app_details` 컬럼은 그대로 추가

In [11]:
appdetails = pd.read_csv('../../../data/raw/steam_app_details.csv')

common_cols_appdetails = sorted(set(df_f.columns) & set(appdetails.columns))
print('games vs app_details 공통 컬럼:', common_cols_appdetails)

replace_cols2    = ['name', 'developers', 'genres', 'release_date']
appdetails_replace = appdetails[['appid'] + replace_cols2]
appdetails_extra   = appdetails.drop(columns=replace_cols2)

df_f = df_f.merge(appdetails_replace, on='appid', how='left', suffixes=('_old', ''))
for col in replace_cols2:
    df_f[col] = df_f[col].combine_first(df_f[f'{col}_old'])
    df_f = df_f.drop(columns=[f'{col}_old'])

df_f = df_f.merge(appdetails_extra, on='appid', how='left')

print('병합 결과 shape:', df_f.shape)
print('컬럼:', df_f.columns.tolist())

games vs app_details 공통 컬럼: ['appid', 'developers', 'genres', 'name', 'release_date']
병합 결과 shape: (9692, 36)
컬럼: ['appid', 'owners', 'positive', 'negative', 'price', 'ccu', 'total_reviews', 'review_score', 'review_score_desc', 'name', 'developers', 'genres', 'release_date', 'type', 'is_free', 'controller_support', 'short_description', 'supported_languages', 'publishers', 'categories', 'coming_soon', 'currency', 'initial', 'final', 'discount_percent', 'initial_formatted', 'final_formatted', 'windows', 'mac', 'linux', 'recommendations_total', 'metacritic_score', 'metacritic_url', 'achievements_total', 'header_image', 'website']


# 데이터 저장

In [12]:
out_path = '../../../data/raw/steam_indie_games.csv'
df_f.to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_f):,}개)')

저장 완료 → ../../../data/raw/steam_indie_games.csv (9,692개)
